# Animated Recreation of `Untitled0.ipynb`

This notebook recreates the plotting sequence from `/Users/thanawatsornwanee/Downloads/Untitled0.ipynb` using the `visualizer` framework for animated curves and fills. The axis arrows and guide lines are drawn directly on the matplotlib axes so their geometry, color, and linewidth stay close to the source notebook.


In [ ]:
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import HTML

plt.rcParams["animation.html"] = "jshtml"

for candidate in (Path.cwd().resolve(), Path.cwd().resolve().parent):
    src_dir = candidate / "src"
    if src_dir.exists():
        sys.path.insert(0, str(src_dir))
        break
else:
    raise RuntimeError("Could not locate the project's src directory.")

from visualizer import (
    Curve,
    CurveStyleTransition,
    DrawTransition,
    EraseFillBetweenTransition,
    EraseTransition,
    FillBetweenArea,
    FillBetweenTransition,
    ParallelTransition,
    Schedule,
)


In [ ]:
def intersect_trim_dec(X, fX, c):
    X, fX = np.asarray(X), np.asarray(fX)
    j = np.argmax(fX <= c)
    if fX[j] > c:
        return None, np.array([]), np.array([])
    if fX[j] == c or j == 0:
        return X[j], X[j:], fX[j:]
    x0, x1 = X[j - 1], X[j]
    y0, y1 = fX[j - 1], fX[j]
    x = x0 + (c - y0) * (x1 - x0) / (y1 - y0)
    return x, np.r_[x, X[j:]], np.r_[c, fX[j:]]


def intersect_trim_inc(X, fX, c):
    X, fX = np.asarray(X), np.asarray(fX)
    j = np.argmax(fX >= c)
    if fX[j] < c:
        return None, np.array([]), np.array([])
    if fX[j] == c or j == 0:
        return X[j], X[j:], fX[j:]
    x0, x1 = X[j - 1], X[j]
    y0, y1 = fX[j - 1], fX[j]
    x = x0 + (c - y0) * (x1 - x0) / (y1 - y0)
    return x, np.r_[x, X[j:]], np.r_[c, fX[j:]]


np.random.seed(626)
N = 10000
epsilon = 1e-2
uprimecut = 30
g = np.random.randn(N) * 1 / np.sqrt(N)
G = np.cumsum(g)
GG = np.cumsum(epsilon + np.power(np.abs(G), 1.7)) / np.sqrt(N)
X = np.linspace(0, 1, N)
U_prime = 10 - GG + 1 / (X + 0.02)
U = np.cumsum(U_prime) / N - 2
U /= np.max(U)
_, X1, U = intersect_trim_inc(X, U, 0)
U_prime /= uprimecut
U_prime_old = U_prime.copy()
_, X2, U_prime = intersect_trim_dec(X, U_prime, 1)

np.random.seed(626)
g = np.random.randn(N) * 1 / np.sqrt(N)
G = np.cumsum(g)
GG = np.cumsum(G) / np.sqrt(N)
Y = 2 - GG
Y = Y[::-1] / 15
_, XY, Ynew = intersect_trim_inc(X, Y, -0.1)
Uprime1 = U_prime_old + Y
_, X21, Uprime1 = intersect_trim_dec(X, Uprime1, 1)

np.random.seed(60)
g = np.random.randn(N) * 1 / np.sqrt(N)
G = np.cumsum(g)
GG = np.cumsum(epsilon + np.power(np.abs(G), 1.7)) / np.sqrt(N) / np.sqrt(N)
Ymonotone = (0.03 - GG) * 2.7
_, X22, Uprime2 = intersect_trim_dec(X, U_prime_old + Ymonotone, 1)

startup = 0.52
startdown = -0.1
neww = 0.49


In [ ]:
def marker_curve(curve_id, x, y, *, marker, color, size=150, zorder=20):
    return Curve(
        curve_id,
        np.array([x]),
        np.array([y]),
        color=color,
        linestyle="None",
        line_kwargs={
            "marker": marker,
            "markersize": float(np.sqrt(size)),
            "zorder": zorder,
        },
    )


def horizontal_curve(curve_id, level):
    return Curve(curve_id, X, np.full_like(X, level), color="black")


def decorate_single_axis(ax):
    arrow_move = 0.03
    scaling = 0.55
    ax.plot(
        [0, 0, 1.05 + scaling * (arrow_move - 0.01)],
        [1.05 + arrow_move - 0.01, 0, 0],
        color="black",
        linewidth=2,
    )
    ax.plot([1.05, 1.05 + scaling * arrow_move, 1.05], [arrow_move, 0, -arrow_move], color="black", linewidth=2)
    ax.plot([scaling * arrow_move, 0, -scaling * arrow_move], [1.05, 1.05 + arrow_move, 1.05], color="black", linewidth=2)


def decorate_split_axes(ax):
    arrow_move = 0.03
    scaling = 0.55
    ax.plot(
        [0, 0, 1.05 + scaling * (arrow_move - 0.01)],
        [startup + neww + arrow_move + 0.05 - 0.01, startup, startup],
        color="black",
        linewidth=2,
    )
    ax.plot([1.05, 1.05 + scaling * arrow_move, 1.05], [startup + arrow_move, startup, startup - arrow_move], color="black", linewidth=2)
    ax.plot([scaling * arrow_move, 0, -scaling * arrow_move], [1.05, 1.05 + arrow_move, 1.05], color="black", linewidth=2)
    ax.plot(
        [0, 0, 1.05 + scaling * (arrow_move - 0.01)],
        [startdown + neww + 0.05 + arrow_move - 0.01, startdown, startdown],
        color="black",
        linewidth=2,
    )
    ax.plot([1.05, 1.05 + scaling * arrow_move, 1.05], [startdown + arrow_move, startdown, startdown - arrow_move], color="black", linewidth=2)
    ax.plot(
        [scaling * arrow_move, 0, -scaling * arrow_move],
        [startdown + neww + 0.05, startdown + neww + 0.05 + arrow_move, startdown + neww + 0.05],
        color="black",
        linewidth=2,
    )


def render_animation(schedule, *, decorate=None, title=None, xlim=(-0.2, 1.2), ylim=(-0.2, 1.2), figsize=(10, 6), fps=24):
    fig, ax = plt.subplots(figsize=figsize)
    ax.set_xlim(*xlim)
    ax.set_ylim(*ylim)
    if decorate is not None:
        decorate(ax)
    anim = schedule.build_animation(fig=fig, ax=ax, fps=fps, xlim=xlim, ylim=ylim, title=title)
    plt.close(fig)
    return HTML(anim.to_jshtml())


In [ ]:
# Animation 1: the red U curve on the custom axis (source cell 3)
schedule = Schedule()
schedule.add(
    DrawTransition(Curve("u_curve", X1, U, linewidth=2, color="red"), show_pointer=False),
    duration=1.6,
)
render_animation(schedule, decorate=decorate_single_axis, title="U on the original axis")


In [ ]:
# Animation 2: the split red/blue view with two axes (source cell 4)
schedule = Schedule()
schedule.add(
    ParallelTransition(
        (
            DrawTransition(Curve("u_top", X1, startup + U * neww, linewidth=2, color="red"), show_pointer=False),
            DrawTransition(Curve("uprime_bottom", X2, startdown + U_prime * neww, linewidth=2, color="blue"), show_pointer=False),
        )
    ),
    duration=1.8,
)
render_animation(schedule, decorate=decorate_split_axes, title="Split view of U and U'")


In [ ]:
# Animation 3: the main U' highlight sequence (source cells 5-15)
schedule = Schedule()
schedule.add(
    DrawTransition(Curve("uprime_blue", X2, U_prime, linewidth=2, color="blue"), show_pointer=False),
    duration=1.0,
)
schedule.add(
    ParallelTransition(
        (
            DrawTransition(horizontal_curve("level_low", 0.16), show_pointer=False),
            DrawTransition(horizontal_curve("level_high", 0.46), show_pointer=False),
        )
    ),
    duration=0.35,
)
schedule.add(
    DrawTransition(Curve("y_green", XY, Ynew, linewidth=2, color="green"), show_pointer=False),
    duration=0.9,
)
schedule.add(
    DrawTransition(Curve("uprime_orange", X21, Uprime1, linewidth=2, color="orange"), show_pointer=False),
    duration=1.0,
)
schedule.add(
    ParallelTransition(
        (
            CurveStyleTransition("uprime_blue", alpha=0.0),
            CurveStyleTransition("y_green", alpha=0.0),
        )
    ),
    duration=0.35,
)
schedule.add(EraseTransition("uprime_blue"), duration=0.0)
schedule.add(EraseTransition("y_green"), duration=0.0)
schedule.add(DrawTransition(marker_curve("anchor_upper", X21[5400], Uprime1[5400], marker=6, color="black"), show_pointer=False), duration=0.0)
schedule.add(
    FillBetweenTransition(
        FillBetweenArea("upper_fill_red_a", X21[5400:5920], Uprime1[5400:5920], 0.46, color="red", alpha=0.2, linewidth=0.0)
    ),
    duration=0.7,
)
schedule.add(DrawTransition(marker_curve("upper_probe_a", X21[5920], Uprime1[5920], marker=6, color="blue"), show_pointer=False), duration=0.0)
schedule.add_break(0.2)
schedule.add(EraseFillBetweenTransition("upper_fill_red_a"), duration=0.0)
schedule.add(EraseTransition("upper_probe_a"), duration=0.0)
schedule.add(
    ParallelTransition(
        (
            FillBetweenTransition(
                FillBetweenArea("upper_fill_red_b", X21[5400:6120], Uprime1[5400:6120], 0.46, color="red", alpha=0.2, linewidth=0.0)
            ),
            FillBetweenTransition(
                FillBetweenArea("upper_fill_green", X21[6120:7400], Uprime1[6120:7400], 0.46, color="green", alpha=0.2, linewidth=0.0)
            ),
            DrawTransition(marker_curve("upper_probe_b", X21[7400], Uprime1[7400], marker=7, color="blue"), show_pointer=False),
        )
    ),
    duration=0.9,
)
schedule.add_break(0.2)
schedule.add(EraseFillBetweenTransition("upper_fill_red_b"), duration=0.0)
schedule.add(EraseFillBetweenTransition("upper_fill_green"), duration=0.0)
schedule.add(EraseTransition("upper_probe_b"), duration=0.0)
schedule.add(EraseTransition("anchor_upper"), duration=0.0)
schedule.add(DrawTransition(marker_curve("anchor_lower", X21[5400], Uprime1[5400], marker=7, color="black"), show_pointer=False), duration=0.0)
schedule.add(
    FillBetweenTransition(
        FillBetweenArea("lower_fill_red_a", X21[4000:5400], Uprime1[4000:5400], 0.16, color="red", alpha=0.2, linewidth=0.0)
    ),
    duration=0.8,
)
schedule.add(DrawTransition(marker_curve("lower_probe_a", X21[4000], Uprime1[4000], marker=7, color="blue"), show_pointer=False), duration=0.0)
schedule.add_break(0.2)
schedule.add(EraseFillBetweenTransition("lower_fill_red_a"), duration=0.0)
schedule.add(EraseTransition("lower_probe_a"), duration=0.0)
schedule.add(
    ParallelTransition(
        (
            FillBetweenTransition(
                FillBetweenArea("lower_fill_red_b", X21[3400:5400], Uprime1[3400:5400], 0.16, color="red", alpha=0.2, linewidth=0.0)
            ),
            FillBetweenTransition(
                FillBetweenArea("lower_fill_green_a", X21[2300:3400], Uprime1[2300:3400], 0.16, color="green", alpha=0.2, linewidth=0.0)
            ),
            DrawTransition(marker_curve("lower_probe_b", X21[2300], Uprime1[2300], marker=6, color="blue"), show_pointer=False),
        )
    ),
    duration=0.9,
)
schedule.add_break(0.2)
schedule.add(EraseFillBetweenTransition("lower_fill_red_b"), duration=0.0)
schedule.add(EraseFillBetweenTransition("lower_fill_green_a"), duration=0.0)
schedule.add(EraseTransition("lower_probe_b"), duration=0.0)
schedule.add(
    ParallelTransition(
        (
            FillBetweenTransition(
                FillBetweenArea("lower_fill_red_c", X21[3400:5400], Uprime1[3400:5400], 0.16, color="red", alpha=0.2, linewidth=0.0)
            ),
            FillBetweenTransition(
                FillBetweenArea("lower_fill_green_b", X21[1700:3400], Uprime1[1700:3400], 0.16, color="green", alpha=0.2, linewidth=0.0)
            ),
            FillBetweenTransition(
                FillBetweenArea("lower_fill_red_d", X21[100:1700], Uprime1[100:1700], 0.16, color="red", alpha=0.2, linewidth=0.0)
            ),
            DrawTransition(marker_curve("lower_probe_c", X21[100], Uprime1[100], marker=7, color="blue"), show_pointer=False),
        )
    ),
    duration=1.0,
)
render_animation(schedule, decorate=decorate_single_axis, title="Main highlight sequence")


In [ ]:
# Animation 4: the monotone comparison (source cells 16-18)
schedule = Schedule()
schedule.add(
    DrawTransition(Curve("uprime_blue", X2, U_prime, linewidth=2, color="blue"), show_pointer=False),
    duration=1.0,
)
schedule.add(
    ParallelTransition(
        (
            DrawTransition(horizontal_curve("level_low", 0.16), show_pointer=False),
            DrawTransition(horizontal_curve("level_high", 0.46), show_pointer=False),
        )
    ),
    duration=0.35,
)
schedule.add(
    DrawTransition(Curve("monotone_green", X, Ymonotone, linewidth=2, color="green"), show_pointer=False),
    duration=0.9,
)
schedule.add(
    DrawTransition(Curve("uprime_orange", X22, Uprime2, linewidth=2, color="orange"), show_pointer=False),
    duration=1.0,
)
render_animation(schedule, decorate=decorate_single_axis, title="Monotone comparison")
